# Build the trip + ping dataset for a single bus (2023)

**Scope**: exploratory only, `explore/vehicle-trip-matching` branch,
queries `silver` directly. This notebook does exactly one thing: build a
dataset and save it. No matching verdicts, no GTFS, no charts.

For one AFC `vehicle_number`:
1. every trip it ran in 2023, per `silver.afc_boardings`
2. the `device_id` `silver.dictionary_device` currently claims for it
3. every AVL ping from that device whose timestamp falls inside a trip's
   `[trip_opened_at, trip_closed_at]` window — matched on **device identity
   and timestamp only**, nothing else assumed

In [10]:
from __future__ import annotations

import datetime as dt
from pathlib import Path

import polars as pl
import psycopg

from opa_database.config import settings

VEHICLE_NUMBER = "14922"
YEAR_START = dt.date(2023, 1, 1)
YEAR_END = dt.date(2024, 1, 1)
# Robust to the two common execution cwds: repo root (terminal / `jupyter
# notebook` launched from root) or this notebook's own directory (VS
# Code's Jupyter extension default) -- descend into notebooks/ only if
# it's actually a subdirectory of cwd, otherwise assume cwd already is it.
NOTEBOOK_DIR = Path("notebooks") if Path("notebooks").is_dir() else Path()
OUTPUT_DIR = NOTEBOOK_DIR / "data"
OUTPUT_PATH = OUTPUT_DIR / f"{VEHICLE_NUMBER}_trips_pings_2023.parquet"

conn = psycopg.connect(settings.silver_dsn)

## 1. The dictionary's claim for this bus

In [ ]:
dict_row = pl.read_database(
    "select vehicle_number, device_id from silver.dictionary_device "
    "where vehicle_number = %(v)s",
    conn,
    execute_options={"params": {"v": VEHICLE_NUMBER}},
)
if dict_row.height == 0:
    msg = f"no dictionary_device entry for vehicle_number {VEHICLE_NUMBER!r}"
    raise ValueError(msg)
device_id = dict_row["device_id"][0]
dict_row

## 2. Every AFC trip for this bus in 2023

One row per distinct `(line_number, line_shift, line_opened_at,
line_closed_at, trip_opened_at, trip_closed_at)` — the same collapse the
architecture doc describes, since `afc_boardings` repeats these on every
boarding row. Filtered on `trip_opened_at` (the actual service time), not
`dump_date` (the bronze partition key, which is the dump file's arrival
date and can lag the service it describes by weeks) — this is
calendar-year 2023 as it actually happened, not as it was uploaded.

In [12]:
afc_raw = pl.read_database(
    """
    select line_number, line_shift, line_opened_at, line_closed_at,
           trip_opened_at, trip_closed_at
    from silver.afc_boardings
    where vehicle_number = %(v)s
      and trip_opened_at >= %(start)s and trip_opened_at < %(end)s
    """,
    conn,
    execute_options={
        "params": {"v": VEHICLE_NUMBER, "start": YEAR_START, "end": YEAR_END}
    },
)
trips = afc_raw.unique(
    subset=[
        "line_number",
        "line_shift",
        "line_opened_at",
        "line_closed_at",
        "trip_opened_at",
        "trip_closed_at",
    ]
).sort("trip_opened_at")
print(f"{afc_raw.height} boardings -> {trips.height} distinct trips")
trips.head()

179026 boardings -> 5145 distinct trips


line_number,line_shift,line_opened_at,line_closed_at,trip_opened_at,trip_closed_at
str,i64,"datetime[μs, Etc/UTC]","datetime[μs, Etc/UTC]","datetime[μs, Etc/UTC]","datetime[μs, Etc/UTC]"
"""76""",2,2022-12-31 17:58:08 UTC,2023-01-01 02:16:29 UTC,2023-01-01 00:11:55 UTC,2023-01-01 01:17:09 UTC
"""76""",2,2022-12-31 17:58:08 UTC,2023-01-01 02:16:29 UTC,2023-01-01 01:17:15 UTC,2023-01-01 02:06:12 UTC
"""76""",2,2022-12-31 17:58:08 UTC,2023-01-01 02:16:29 UTC,2023-01-01 02:06:16 UTC,2023-01-01 02:16:14 UTC
"""76""",1,2023-01-01 09:08:49 UTC,2023-01-01 14:04:08 UTC,2023-01-01 09:12:14 UTC,2023-01-01 10:30:26 UTC
"""76""",1,2023-01-01 09:08:49 UTC,2023-01-01 14:04:08 UTC,2023-01-01 10:30:30 UTC,2023-01-01 11:34:12 UTC


## 3. Every AVL ping from the mapped device in 2023

In [13]:
avl = pl.read_database(
    """
    select metric_timestamp, latitude, longitude, route_code, direction, odometer, speed
    from silver.avl_pings
    where device_id = %(d)s
      and metric_timestamp >= %(start)s and metric_timestamp < %(end)s
    order by metric_timestamp
    """,
    conn,
    execute_options={
        "params": {
            "d": device_id,
            "start": dt.datetime.combine(YEAR_START, dt.time(), tzinfo=dt.UTC),
            "end": dt.datetime.combine(YEAR_END, dt.time(), tzinfo=dt.UTC),
        }
    },
    schema_overrides={"latitude": pl.Float64, "longitude": pl.Float64},
)
print(f"{avl.height} pings for device {device_id}")
avl.head()

1667099 pings for device ep1-428115079


metric_timestamp,latitude,longitude,route_code,direction,odometer,speed
"datetime[μs, Etc/UTC]",f64,f64,i64,i64,i64,i64
2023-01-01 00:11:54 UTC,-3.772991,-38.607561,76,62,16678429,0
2023-01-01 00:12:34 UTC,-3.772991,-38.607558,76,62,16678429,5
2023-01-01 00:12:37 UTC,-3.772991,-38.607561,76,0,16678429,0
2023-01-01 00:13:09 UTC,-3.772991,-38.607561,76,62,16678429,0
2023-01-01 00:13:27 UTC,-3.773027,-38.607591,76,198,16678432,6


## 4. Match: one row per ping, tagged with the trip it falls inside

A ping belongs to a trip when the device matches (already true — we only
pulled this one device's pings) and its timestamp falls inside that trip's
`[trip_opened_at, trip_closed_at]` window. Trips for one vehicle don't
overlap (checked below), so "the most recent trip that opened before this
ping, provided it hadn't closed yet" identifies the window uniquely — an
as-of join expresses exactly that, vectorized, instead of a slow
row-by-row scan over ~1.6M pings.

In [ ]:
overlaps = trips.with_columns(
    pl.col("trip_opened_at").shift(-1).alias("next_open")
).filter(pl.col("next_open") < pl.col("trip_closed_at"))
print(f"trips overlapping the next trip: {overlaps.height} (should be 0)")

dataset = (
    avl.join_asof(
        trips,
        left_on="metric_timestamp",
        right_on="trip_opened_at",
        strategy="backward",
    )
    .filter(pl.col("metric_timestamp") <= pl.col("trip_closed_at"))
    .with_columns(
        pl.lit(VEHICLE_NUMBER).alias("vehicle_number"),
        pl.lit(device_id).alias("device_id"),
    )
    .select(
        "vehicle_number",
        "device_id",
        "line_number",
        "line_shift",
        "line_opened_at",
        "line_closed_at",
        "trip_opened_at",
        "trip_closed_at",
        pl.col("metric_timestamp").alias("ping_timestamp"),
        "latitude",
        "longitude",
        "route_code",
        "direction",
        "odometer",
        "speed",
    )
    .sort("trip_opened_at", "ping_timestamp")
)

n_trips_with_pings = dataset.select("trip_opened_at").n_unique()
n_unmatched = avl.height - dataset.height
print(
    f"{dataset.height} ping-rows matched, "
    f"{n_unmatched} pings fell outside every trip window"
)
print(f"{n_trips_with_pings} / {trips.height} trips have at least one matching ping")
dataset.head()

## 5. Save

In [15]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
dataset.write_parquet(OUTPUT_PATH)
print(f"wrote {dataset.height} rows to {OUTPUT_PATH}")

wrote 1510068 rows to data/14922_trips_pings_2023.parquet
